# Chess Module

## Dependencies

Explanatory comments in the following block are added before the command. 

In [32]:
# A python package to store binary files. 
# !pip install dill

# A python package for embedding HTML widgets in Jupyter notebooks
# !pip install ipywidgets

# A python package for embedding HTML widgets in Jupyter lab (different from notebook)
# !pip install jupyterlab_widgets

# A python package for embedding HTML widgets in Jupyter lab (different from notebook)
# Use this if the previous command fails.
# %conda install jupyterlab_widgets

# Chess Package
# !pip install python-chess

# %pip install numpy dill

## Starting Positions

Sample starting positions to test your AI agent. Some of these games have high branch factor which can cause a very slow search when searching deeper than 4.

In [33]:
# Openings to investigate
# CAUTION: Will take time on higher depths ( > 4)
ClosedSicilianDefence = 'rnbqkbnr/pp1ppppp/8/2p5/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 1 2'
ViennaOpenning = 'rnbqkbnr/pppp1ppp/8/4p3/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 1 2'
NimzoIndianDefence = 'rnbqk2r/pppp1ppp/4pn2/8/1bPP4/2N5/PP2PPPP/R1BQKBNR w KQkq - 2 4'
SlavDefence = 'rnbqkbnr/pp2pppp/2p5/3p4/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3'
QueenzGambit = 'rnbqkbnr/ppp1pppp/8/3p4/2PP4/8/PP2PPPP/RNBQKBNR b KQkq c3 0 2'
RuyLopez = 'r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3'

WM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/R1BQKB1R w KQ - 0 1"
BM5Position = "2kr4/1b2b1p1/p2pp3/1p6/5B2/2P1NqP1/PP3PQK/8 b - - 0 1"
BM6Position = "r1b1kr2/bp2qp2/p2p2P1/2n1p3/2Q1P1P1/1P1PN2p/P1PBR2P/2NRKBn1 b q - 0 1"

NotM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/RQB1KB1R w KQ - 0 1"
Zugzwang = "kbK5/pp6/1P6/8/8/8/8/R7 w - - 0 1"
StartingBoard = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
WMK2R = "8/8/8/8/8/4k3/6R1/6RK w - - 0 1"
WMK1REasy = "3k4/R7/8/4K3/8/8/8/8 w - - 0 1"
WMK1RMedium = "8/6K1/8/8/4k3/8/8/6R1 w - - 0 1"
WMK1RHard = "8/8/8/8/4k3/8/8/6RK w - - 0 1"
KingRoom = "8/8/8/8/3pp3/4k3/6R1/6RK w - - 0 1"

WMKQR = "8/8/8/3k4/8/8/8/5RQK w - - 0 1"

DKnightPromotionFork = "8/3KP1k1/5q2/8/8/8/8/8 w - - 0 1"

BTraxler = 'r1bqk2r/pppp1Npp/2n2n2/2b1p3/2B1P3/8/PPPP1PPP/RNBQK2R b KQkq - 0 1'
WAntiTraxler = 'r1bqk2r/pppp1ppp/2n2n2/2b1p1N1/2B1P3/8/PPPP1PPP/RNBQK2R w KQkq - 0 1'
Activity_vs_material = "rn2kbnr/1ppp1pp1/8/8/4P3/2P2N2/PB1P1PPP/RNQ1KB1R b KQkq - 0 5"

BM4SmutheredMate = "rnb2rk1/pp3ppp/1qpp4/8/2B1P1n1/3P1N2/PPP3PP/RNBQR2K b - - 6 10"
BM4SmutheredMate2 ="rnb2rk1/pp3ppp/1qpp4/8/2B3n1/3P1N2/PPP2PPP/RNBQR2K b - - 6 10"

## Imports

In [34]:
import random                                   
import numpy as np                              
import time                                     
import chess                                    

from ChessEnv.ZEIT4150.chess import ChessAgent, ChessGame
from IPython.display import display, HTML, clear_output
from chess import Move, polyglot, Board, Square, SquareSet, WHITE, BLACK

from math import log, sqrt
inf = float('inf')

## Board Scoring

The following functions evaluate the position on the board and produce a score between `-inf` and `+inf`. Positive scores favour the white player (i.e. `chess.WHITE` or `True`). Negative scores favour the black player (i.e. `chess.BLACK` or `False`). Scoring functions can be heavy (1 < t < 50 micro seconds). It can be slower if justified if the move ordering is in effect.

In [35]:
# Attribution\n\nPortions of the board-scoring code in the next cell were generated with assistance from OpenAI ChatGPT (GPT-5.3-Codex), then reviewed and adapted by Brendon Bone.\n\nReference: OpenAI (2026). ChatGPT. https://openai.com/models/chatgpt/

import operator
from functools import reduce

# AI-assisted block starts here (OpenAI ChatGPT, reviewed and adapted by Brendon Bone).
def prepBoard4Eval(b):
    """Prepares list of squares and attacking masks for white and black to be used in other evaluation functions. """
    b.squares = [SquareSet(b.occupied_co[BLACK]),
                 SquareSet(b.occupied_co[WHITE])]
    
    b.piece_masks = [b.pawns, b.knights, b.bishops, b.rooks, b.queens, b.kings]
    
    b.attacking_mask = [
        reduce(operator.__or__, 
               (b.attacks_mask(sq) for sq in b.squares[BLACK]), 0),
        reduce(operator.__or__, 
               (b.attacks_mask(sq) for sq in b.squares[WHITE]), 0)]
    return

def occupation_difference(b):
    """Difference in number of pieces on the board"""
    nWhiteSquares = bin(b.occupied_co[chess.WHITE]).count('1')
    nBlackSquares = bin(b.occupied_co[chess.BLACK]).count('1')
    
    return nWhiteSquares - nBlackSquares

# AI-assisted: piece-value material scoring.
def material_difference(b):
    """Difference in quality of the pieces on the board (e.g. a queen is worth 9 pawns)"""
    piece_values = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0,
    }
    nWhite, nBlack = 0, 0
    for ptype, value in piece_values.items():
        nWhite += value * b.pieces_mask(ptype, chess.WHITE).bit_count()
        nBlack += value * b.pieces_mask(ptype, chess.BLACK).bit_count()
    return nWhite - nBlack

# AI-assisted: bishop pair bonus.
def bishop_difference(b):
    """Difference of bishops on the board with a bonus if the bishop pair is present."""
    n_white_bishops = b.pieces_mask(chess.BISHOP, chess.WHITE).bit_count()
    n_black_bishops = b.pieces_mask(chess.BISHOP, chess.BLACK).bit_count()
    white_bishop_pair = 0.5 if n_white_bishops >= 2 else 0
    black_bishop_pair = 0.5 if n_black_bishops >= 2 else 0
    
    return n_white_bishops + white_bishop_pair - n_black_bishops - black_bishop_pair

# AI-assisted: attacked-piece pressure count.
def attack_difference(b):
    """Difference in attacking pieces for each attacked square."""
    nWhite, nBlack = 0, 0
    for sq in b.squares[chess.BLACK]:
        nWhite += len(b.attackers(chess.WHITE, sq))
    for sq in b.squares[chess.WHITE]:
        nBlack += len(b.attackers(chess.BLACK, sq))
    return nWhite - nBlack

# AI-assisted: threatened-enemy-piece count.
def threat_difference(b):
    """Differnece in pieces attacking the other colour."""
    nWhite = (b.attacking_mask[chess.WHITE] & b.occupied_co[chess.BLACK]).bit_count()
    nBlack = (b.attacking_mask[chess.BLACK] & b.occupied_co[chess.WHITE]).bit_count()
    return nWhite - nBlack

# AI-assisted: legal king moves per side.
def king_mobility(b):
    """Difference in number of escape squares available for the king."""
    def _count_king_moves(board, color):
        board_copy = board.copy(stack=False)
        board_copy.turn = color
        ksq = board_copy.king(color)
        if ksq is None:
            return 0
        return sum(1 for mv in board_copy.legal_moves if mv.from_square == ksq)

    nWhite = _count_king_moves(b, chess.WHITE)
    nBlack = _count_king_moves(b, chess.BLACK)
    return nWhite - nBlack

# AI-assisted: center occupancy (d4, e4, d5, e5).
def center_difference(b):
    """Difference in number of pieces occupying the center"""
    center_squares = (chess.D4, chess.E4, chess.D5, chess.E5)
    nWhite, nBlack = 0, 0
    for sq in center_squares:
        piece = b.piece_at(sq)
        if piece is None:
            continue
        if piece.color == chess.WHITE:
            nWhite += 1
        else:
            nBlack += 1
    return nWhite - nBlack

# AI-assisted: terminal mate score sign handling.
def checkmate(b):
    """Returns a score for the check mate."""
    if not b.is_checkmate():
        return 0
    return -1 if b.turn == chess.WHITE else 1

# AI-assisted: in-check pressure score.
def check(b):
    """Return score for check."""
    if not b.is_check():
        return 0
    return -1 if b.turn == chess.WHITE else 1

def isDraw(b):
    draw = b.can_claim_draw()
    return int(not draw)

# AI-assisted: weighted aggregate scoring function.
def evalBoard(b):
    """Returns the overall score."""
    prepBoard4Eval(b)
    return isDraw(b) * (0
                        + 1*occupation_difference(b)
                        + 2*center_difference(b)
                        + 3*bishop_difference(b)
                        + 4*material_difference(b)
                        + 5*attack_difference(b)
                        + 6*threat_difference(b)
                        + 7*king_mobility(b) 
                        + 9*check(b) 
                        + 1000*checkmate(b)
                        + 0)
# AI-assisted block ends here.



### Move Ordering

The following function is an example of a move ranking function. It takes the board and a move and returns a number based on which we can sort the moves to have a better AlphaBeta pruning. This function should be very fast (i.e. t < 500 nano seconds) because it will be called on every leagal move. 

The role of this function is to order the moves in desendingly to maximise pruning in the search.

In [36]:
def rankMove(b, mv):
    return (0 
            + 1*b.is_zeroing(mv)
            + 2*b.is_en_passant(mv)
            + 3*b.is_castling(mv)
            + 4*b.is_capture(mv)
            # + 5*threats(b, mv)
            + 9*b.gives_check(mv)
            # + 9*b.is_into_checkmate(mv)
            # - 1*b.is_irreversible(mv) 
            + 0)


### Move Ranking Examples

The following examples show how different move types are ranked by the `rankMove` function. Moves with higher scores are explored first during alpha-beta pruning, improving search efficiency.

## Example: Random Agent (Unchanged)

In [37]:
class RandomChessAgent(ChessAgent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        pass
    
    def calcMoves(self, board):
        # Always wrap legal_moves into a list to 
        # unpack moves. Otherwise, it returns 
        # a generator.
        legalMoves = list(board.legal_moves)
        
        N = 2
        
        myMoves = []
        myMoves = np.random.choice(legalMoves, N).tolist()
        # Draw list of moves on the board (blue)
        self.drawMyMoves(myMoves)
        
        # Log relevant info per agent
        self.log2Agent(f"INFO: myMoves={[board.san(mv) for mv in myMoves]}", "green")
        
        # Draw a minitutre version of this board
        # You can use this to visualise current scenario
        # your agent is investigating.
        self.drawBoard(board, "Board1")
        
        # Log info to the margin of the board. This is a shared 
        # logging space between this agent, the opponent agent 
        # and the board itself.
        # self.log2Margin(f"This is an INFO msg!!")
        for mv in myMoves:
            
            # Apply action to the board
            board.push(mv)
            
            # Query leagal moves after action`mv` applied
            legalMoves = list(board.legal_moves)
            
            if len(legalMoves) >= N:
                oppMoves = np.random.choice(legalMoves, 2).tolist()
                
                # Log relevant info per agent
                self.log2Agent(f"INFO: oppMoves={[board.san(mv) for mv in oppMoves]}", "red")
                
                # Draw list of opponent moves on the board (red)
                self.drawOpponentMoves(oppMoves)
            board.pop()
            pass
        
        # Draw a mask board showing king's reach
        self.drawBoard(board.attacks(board.king(board.turn)), 
                       "King")
        
        # Draw a mask board showing opponent's king's reach
        self.drawBoard(board.attacks(board.king(not board.turn)), 
                       "Other King")
        
        self.drawBoard(board, "Mask 3")
        self.drawBoard(board.attackers(chess.WHITE, chess.F5), 
                       "f5 Attackers")
        
        self.drawBoard(board.attackers(chess.WHITE, chess.E5), 
                       "e5 Attackers")
        
        self.drawBoard(board.attackers(chess.WHITE, chess.D5), 
                       "d5 Attackers")
        
        self.drawBoard(chess.SquareSet(chess.BB_LIGHT_SQUARES & chess.BB_RANK_3), 
                       "3rd Rank Light Squares")
        
        # Scoring function check base class for more info 
        score = self.scoreFn(self.board)
        self.log2Agent(f"{score=}", "black")
        
        # A sleeping timer to slow things down (only releven when visualising random agents)
        time.sleep(self.dtime)
        return myMoves[-1]
        
    def getAction(self, board):
        return self.calcMoves(board)
    pass

## MinMax Adversarial state space search algorithm (Task 1a)

To implement MinMax this video: https://www.youtube.com/watch?v=l-hh51ncgDI

Along with this Github reposity: https://github.com/nadeem4/chess_engine_using_python/tree/main
Inspired my coding

In [ ]:
# This was copied from the random agent and adapted to implement a minmax search.

class MinMaxAgent(ChessAgent):
    def __init__(self, maxDepth=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth

    def _minmax(self, board, depth, maximizing_white):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None

        legal_moves = list(board.legal_moves)

        if maximizing_white:
            best_score = -inf

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._minmax(board, depth - 1, False)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv

            return best_score, best_move

        best_score = inf

        for mv in legal_moves:
            board.push(mv)
            score, _ = self._minmax(board, depth - 1, True)
            board.pop()

            if score < best_score:
                best_score = score
                best_move = mv

        return best_score, best_move

    def calcMoves(self, board):
        maximizing_white = bool(board.turn == chess.WHITE)
        score, best_move = self._minmax(board, self.maxDepth, maximizing_white)

        self.drawMyMoves([best_move])
        self.log2Agent(
            f"INFO: MinMax depth={self.maxDepth}, move={board.san(best_move)}, score={score}",
            "green",
        )

        time.sleep(self.dtime)
        return best_move

    def getAction(self, board):
        return self.calcMoves(board)

## Enhanced MinMax by using AlphaBeta pruning (Task 1B)

In [ ]:
class MinMaxAgent_AlphaBetaPruning(ChessAgent):
    # This was copied from the random agent and adapted to implement a minmax search.
    def __init__(self, maxDepth=2, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.maxDepth = maxDepth

    # AlphaBeta pruning implementation.
    def _alphabeta(self, board, depth, alpha, beta, maximizing_white):
        if depth == 0 or board.is_game_over(claim_draw=True):
            return self.scoreFn(board), None

        legal_moves = list(board.legal_moves)
        if not legal_moves:
            return self.scoreFn(board), None

        if maximizing_white:
            best_score = -inf
            best_move = legal_moves[0]

            for mv in legal_moves:
                board.push(mv)
                score, _ = self._alphabeta(board, depth - 1, alpha, beta, False)
                board.pop()

                if score > best_score:
                    best_score = score
                    best_move = mv

                alpha = max(alpha, best_score)
                if alpha >= beta:
                    break

            return best_score, best_move

        best_score = inf
        best_move = legal_moves[0]

        for mv in legal_moves:
            board.push(mv)
            score, _ = self._alphabeta(board, depth - 1, alpha, beta, True)
            board.pop()

            if score < best_score:
                best_score = score
                best_move = mv

            beta = min(beta, best_score)
            if alpha >= beta:
                break

        return best_score, best_move

    def calcMoves(self, board):
        maximizing_white = bool(board.turn == chess.WHITE)
        score, best_move = self._alphabeta(board, self.maxDepth, -inf, inf, maximizing_white)

        self.drawMyMoves([best_move])
        self.log2Agent(
            f"INFO: AlphaBeta depth={self.maxDepth}, move={board.san(best_move)}, score={score}",
            "green",
        )

        time.sleep(self.dtime)
        return best_move

    def getAction(self, board):
        return self.calcMoves(board)


## Game Play

### Test Cases

In [40]:
# Starting board. This is the standard chess game. White moves 1st. 
StartingBoard = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"

# White is trapped in a dicovery check by the knight and the queen. Black's knight 
# forks the king and the queen on F2 forcing the white king to H1. Black then has to
# reject taking the queen on D1 and instead deliver a double check (queen to assist) 
# which forces the white king hide in the corner. Black will then sacrifice the queen 
# on G1 which forces the white Rook to take and trapping its own king even further. 
# Finally, Black delivers check mate with Nf2#.
# This puzzle measures the long term planning which guarantees a win if the agent 
# ignores material gain (ignoring the forked queen on D1) and even sacrificing its 
# own queen on G1.
# 
# Solution (Black checkmates in 4 moves): 10... Nf2+ 11. Kg1 Nh3+ 12. Kh1 Qg1+ 13. Rxg1 Nf2#
BM4SmutheredMate = "rnb2rk1/pp3ppp/1qpp4/8/2B1P1n1/3P1N2/PPP3PP/RNBQR2K b - - 6 10"

# This puzzle, the white knight has to clear the way to the queen to deliver checkmate 
# on H5. The knight has to choose between taking the pawn on G5 or just move to E5. Both 
# moves clears the way to the queen but Ne5 is the correct move because it blocks D7. 
# If not Ne5, Black can clear the D7 square after sacrificing the rook on E6 with a check.
# 
# Solution (White checkmates in 5 moves): 1. Ne5 g4 2. Qxg4 Qxc3+ 3. bxc3 fxe5 4. Qh5+ Rg6 5. Qxg6#
WM5Position = "2nrkbn1/pp1bp3/2qprp1p/6p1/4P2P/2NP1N2/PPP2PP1/R1BQKB1R w KQ - 0 9"

# This puzzle measures precision. The black king is trapped at A8. If White loses 
# the rook, the game ends in a draw. It is white's turn and Black is counting on this 
# to ease the pressure. White needs to waste a move and be forcing at the same time. 
# The wasted move forces Black to chose between two very bad moves and losing the game.
# This tractic is what is known as "Zugzwang". The solution is to move the rook to A6
# forcing Black to take with the pawn and thus allows White to push the B6 pawnto B7 
# and checkmates Black.
# 
# Solution (White checkmates in 2 moves): 53. Ra6 bxa6 54. b7#
BM2Zugzwang = "kbK5/pp6/1P6/8/8/8/8/R7 w - - 0 53"

# This puzzle measures precision. There is one order of moves to solve this puzzle. 
# The white rook on E1 (not C1) must check the black king. The the knight delivers a
# a check on F4. Finally, the white rook on C1 delivers a check mate on c3.
# 
# Solution (White checkmates in 3 moves): 30. Red1+ Ke2 31. Nf4+ Ke3 32. Rc3#
WM3MateNet = "3rr3/p5bp/2nB2p1/2PN4/3P4/3k1P2/P5PP/2R1R1K1 w - - 4 30"

# End-game puzzles aim to measure the efficacy of transposition table and the scoring 
# function. White can win easily with a king and a rook. Black is aiming to draw the 
# game by repetition or 50 moves rules. The white King must move to G6. Black can only 
# go to G8. Finally the white rook delivers mate on A1.
# 
# Solution (White checkmates in 2 moves): 62. Kg6 Kg8 63. Ra8#
WM2EndGameKingRook = "7k/R7/8/5K2/8/8/8/8 w - - 0 62"

# End-game puzzles aim to measure the efficacy of transposition table and the scoring 
# function. White can win easily with a 2 rooks or a rook and a queen. Black is aiming
# to draw the game by repetition or 50 moves rules. The white rooks can perform a 
# ladder mate with rooks checking the black king on A6, B7 and delivers mate on A8.
# 
# Solution (White checkmates in 3 moves): 62. Ra6+ Kh7 63. Rb7+ Kh8 64. Ra8#
WM3EndGame2RooksLadderMate = "8/8/6k1/1R6/8/8/8/R2K4 w - - 0 62"

# This end-game puzzle dictates on white to under promote the pawn to a bishop to avoid
# a stalemate. White pushes the pawn to D8. Black's only moveable piece is the knight
# and the correct move is knight to F8. Now if white promotes to a queen or a rook, the
# black knight is pinned and thus black has no legal moves but no in check (i.e. stalemate)
# the game is drawn. Thus white must promote to a bishop on D8 and thus the black knight is 
# available to move (i.e. stalemate avoided) and the game continues with mate in 2.
# 
# Solution (White checkmates in 4 moves): 64. d7 Nf8 65. d8=B Ne6 66. Bf6+ Ng7 67. Bxg7#
WM4UnderPromotion = "7k/5B1n/3P3K/8/8/8/8/8 w - - 0 64"



### Game Play

In [ ]:
white = MinMaxAgent(name="White", scoreFn=evalBoard, dtime=0, maxDepth=3)
black = RandomChessAgent(name="Black", scoreFn=evalBoard, dtime=0)

game = ChessGame(fen=StartingBoard, vizSize=350) 
clear_output(wait=True)

for i in range(10):
    
    game.addGameInfo(f"Game {i:03d}: {white.name} vs. {black.name}")
    game.playGame(white, black, visualise=True)
    pgn = game.saveGame(f"Game__{i} - {white.name} vs. {black.name}.pgn")
    clear_output(wait=True)
    game.resetGame()
    

AttributeError: 'MinMaxAgent' object has no attribute 'logAverageDecisionTime'